# DICE ITC Results Notebook

This notebook is the public, portable entry point for reproducing the DICE ITC results. Run it top to bottom.

It regenerates:
- the core analysis tables and figures
- the full DICE paper results
- the workload-holdout appendix results
- the `results_itc_paper/` and `results_itc_appendix/` bundles
- the reproducibility manifest

The notebook is organized in paper order:
1. execution guard and end-to-end run
2. benign/anomaly separation and core full-pipeline metrics
3. calibrated reliability without per-workload tuning
4. digital-twin observability, tier, and mechanism dashboards
5. workload-holdout robustness, bootstrap confidence intervals, and reproducibility artifacts

Methodology reflected here:
1. Benign-only regime-conditioned micro-twin heads across Tier-0, Tier-0/1, and Tier-0/1/2 observation availability.
2. Online residualization with fixed block summaries.
3. Sequential conformal decisioning with persistent alerts.
4. Mechanism-level diagnosis from grouped residual evidence.
5. Workload-holdout robustness as a portable workload/software-drift proxy.
6. Reduced-observability robustness across tier subsets.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile
from time import perf_counter

import pandas as pd
from IPython.display import Image, display
from sklearn.metrics import average_precision_score, roc_auc_score

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


In [ ]:
def resolve_repo_root(start: Path) -> Path:
    for base in [start, *start.parents]:
        if (base / 'tools' / 'run_results_pipeline.py').exists() and (base / 'data generation').exists():
            return base
    raise RuntimeError('Could not locate the DICE repository root from the current working directory.')


def portable_env() -> dict[str, str]:
    env = os.environ.copy()
    env['MPLCONFIGDIR'] = env.get('MPLCONFIGDIR', tempfile.mkdtemp(prefix='dice-mpl-'))
    env['MPLBACKEND'] = 'Agg'
    env['OPENBLAS_NUM_THREADS'] = '1'
    env['OMP_NUM_THREADS'] = '1'
    env['MKL_NUM_THREADS'] = '1'
    env['NUMEXPR_NUM_THREADS'] = '1'
    env['VECLIB_MAXIMUM_THREADS'] = '1'
    env['BLIS_NUM_THREADS'] = '1'
    env['PYTHONHASHSEED'] = '0'
    return env


REPO_ROOT = resolve_repo_root(Path.cwd().resolve())
DATASET_ROOT = REPO_ROOT / 'data generation' / 'dataset' / 'ITC_M2Pro_DATA'
_INTERNAL_PIPELINE = REPO_ROOT / 'tools' / 'run_results_pipeline.py'
OUT = DATASET_ROOT / 'results_analysis'
FIG = OUT / 'figures'
OUT_FULL = DATASET_ROOT / 'results_dice_full'
OUT_HOLDOUT = DATASET_ROOT / 'results_dice_full_holdout'
OUT_PAPER = DATASET_ROOT / 'results_itc_paper'
OUT_APPENDIX = DATASET_ROOT / 'results_itc_appendix'
MANIFEST = DATASET_ROOT / 'results_portable' / 'run_manifest.json'

print('REPO_ROOT    :', REPO_ROOT)
print('DATASET_ROOT :', DATASET_ROOT)


## Execution Guard

The notebook should be launched from the intended DICE clone, not from a stale copy in `Trash` or another transient folder.

The next cell validates the repository location, confirms the released dataset is available, and prepares the main paper and appendix output folders.


In [ ]:
if '.Trash' in str(REPO_ROOT):
    raise RuntimeError(
        'This notebook was launched from a Trash clone. Reopen it from your intended DICE repository checkout.'
    )
assert DATASET_ROOT.exists(), f'Missing dataset root: {DATASET_ROOT}'

PAPER_FULL = OUT_PAPER / 'full'
PAPER_FIG = PAPER_FULL / 'figures'
APPENDIX_FULL = OUT_APPENDIX / 'full'
NOTEBOOK_RUNTIME = OUT_PAPER / 'runtime_summary.json'

for path in [OUT_PAPER, OUT_APPENDIX, PAPER_FULL, PAPER_FIG, APPENDIX_FULL]:
    path.mkdir(parents=True, exist_ok=True)

print('Validated repository root :', REPO_ROOT)
print('Validated dataset root    :', DATASET_ROOT)
print('Main paper output folder  :', PAPER_FULL)
print('Appendix output folder    :', APPENDIX_FULL)


## Run End-to-End

Set `RUN_END_TO_END=True` and execute the next cell. The notebook will orchestrate the internal backend automatically and write the paper and appendix bundles.


In [ ]:
RUN_END_TO_END = True
INCLUDE_TUNING = False

runtime_start = perf_counter()

if RUN_END_TO_END:
    cmd = [sys.executable, str(_INTERNAL_PIPELINE), '--package_itc']
    if INCLUDE_TUNING:
        cmd.append('--run_tuning')
    proc = subprocess.run(
        cmd,
        cwd=str(REPO_ROOT),
        env=portable_env(),
        capture_output=True,
        text=True,
    )
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f'Internal DICE pipeline failed with exit code {proc.returncode}')
else:
    print('Skipped end-to-end run. Set RUN_END_TO_END=True to execute.')

runtime_seconds = round(perf_counter() - runtime_start, 2)
runtime_summary = {
    'runtime_seconds': runtime_seconds,
    'runtime_minutes': round(runtime_seconds / 60.0, 2),
    'include_tuning': INCLUDE_TUNING,
    'repo_root': str(REPO_ROOT),
    'dataset_root': str(DATASET_ROOT),
}
NOTEBOOK_RUNTIME.write_text(json.dumps(runtime_summary, indent=2))
print('Notebook runtime summary:')
display(pd.DataFrame([runtime_summary]))


## ITC Artifact Roadmap

The rest of the notebook is grouped into the main-paper and appendix outputs that reviewers usually look for:
- main-paper separability, calibrated reliability, diagnosis, and digital-twin dashboards
- appendix holdout robustness, prediction review tables, bootstrap confidence intervals, and reproducibility files


## Core Analysis Outputs

These are the tier-level analysis tables and figures used to describe separability and dataset coverage.


In [ ]:
overall = pd.read_csv(OUT / 'table_overall_metrics.csv')
stressor = pd.read_csv(OUT / 'table_stressor_metrics.csv')
workload = pd.read_csv(OUT / 'table_workload_summary.csv')
features = pd.read_csv(OUT / 'table_feature_inventory.csv')
quality = pd.read_csv(OUT / 'table_case_quality.csv')

print('Overall metrics')
display(overall)

print('Per-stressor metrics')
display(stressor)

print('Workload summary')
display(workload)

print('Feature inventory')
display(features[['tier_name', 'n_features_common', 'n_features_union']])

print('Case quality snapshot')
display(quality.head())


In [ ]:
for path in [
    FIG / 'fig_heatmap_pr_auc.png',
    FIG / 'fig_run_score_distributions.png',
    FIG / 'fig_af_timeseries_tier2.png',
]:
    print(path)
    if path.exists():
        display(Image(filename=str(path)))


## Methodology-Oriented Full Results

These outputs align with the preferred methodology: benign-only modeling, sequential decisioning, mechanism-level diagnosis, and robustness across observation heads.


In [ ]:
overall_full = pd.read_csv(OUT_FULL / 'overall_metrics.csv')
stressor_full = pd.read_csv(OUT_FULL / 'stressor_metrics_final_config.csv')
sequential = pd.read_csv(OUT_FULL / 'sequential_metrics.csv')
diagnosis = pd.read_csv(OUT_FULL / 'stressor_diagnosis_metrics.csv')
mechanism = pd.read_csv(OUT_FULL / 'mechanism_group_summary.csv')
tier_contrib = pd.read_csv(OUT_FULL / 'stressor_tier_contributions.csv')

print('Overall full-pipeline metrics')
display(overall_full)

print('Sequential decision metrics')
display(sequential)

print('Mechanism-group diagnosis metrics')
display(diagnosis)

print('Mechanism-group summary by stressor')
display(mechanism)

print('Tier contribution summary by stressor')
display(tier_contrib)

row_final = overall_full[overall_full['config'] == 'tier0_tier1_tier2'].iloc[0]
seq_final = sequential[sequential['config'] == 'tier0_tier1_tier2'].iloc[0]
diag_final = diagnosis[diagnosis['config'] == 'tier0_tier1_tier2'].iloc[0]
print('Final config (Tier-0 + Tier-1 + Tier-2)')
print('ROC-AUC              :', round(float(row_final['roc_auc_wc']), 4))
print('AUC-PR               :', round(float(row_final['pr_auc_wc']), 4))
print('Benign alert rate    :', round(float(seq_final['benign_run_alert_rate']), 4))
print('Median time-to-detect:', round(float(seq_final['median_time_to_detect_s']), 2))
print('Top-1 diagnosis acc  :', round(float(diag_final['top1_acc']), 4))
print('Top-2 diagnosis acc  :', round(float(diag_final['top2_acc']), 4))


In [ ]:
for path in [
    OUT_FULL / 'figures' / 'fig_roc_pr_by_config_wc.png',
    OUT_FULL / 'figures' / 'fig_detection_latency.png',
    OUT_FULL / 'figures' / 'fig_mechanism_group_summary.png',
    OUT_FULL / 'figures' / 'fig_stressor_confusion_matrix.png',
    OUT_FULL / 'figures' / 'fig_stressor_tier_contributions.png',
]:
    print(path)
    if path.exists():
        display(Image(filename=str(path)))


## Benign/Anomaly Separation and Prediction Review

These tables establish the core paper story before any more elaborate diagnosis: benign and anomaly runs should separate cleanly, and reviewers should be able to inspect representative detected anomalies, false alarms, and misses.


In [ ]:
case_pred = pd.read_csv(OUT_FULL / 'case_predictions.csv')

sep_summary = (
    case_pred.groupby(['config', 'label'], sort=False)['run_score_wc']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .reset_index()
)
sep_summary['label_name'] = sep_summary['label'].map({0: 'benign', 1: 'anomaly'})
sep_summary.to_csv(PAPER_FULL / 'benign_anomaly_separation_summary.csv', index=False)

review_cols = ['config', 'case_id', 'workload', 'stressor', 'label', 'run_score_wc', 'run_alert', 'min_pvalue', 'time_to_detect_s']
tp = case_pred[(case_pred['label'] == 1) & (case_pred['run_alert'] == 1)][review_cols].sort_values('run_score_wc', ascending=False).head(10)
fp = case_pred[(case_pred['label'] == 0) & (case_pred['run_alert'] == 1)][review_cols].sort_values('run_score_wc', ascending=False).head(10)
fn = case_pred[(case_pred['label'] == 1) & (case_pred['run_alert'] == 0)][review_cols].sort_values('run_score_wc', ascending=True).head(10)

tp.to_csv(APPENDIX_FULL / 'prediction_review_true_positives.csv', index=False)
fp.to_csv(APPENDIX_FULL / 'prediction_review_false_positives.csv', index=False)
fn.to_csv(APPENDIX_FULL / 'prediction_review_false_negatives.csv', index=False)

print('Benign/anomaly separation summary')
display(sep_summary[['config', 'label_name', 'count', 'mean', 'median', 'min', 'max']])

print('Top detected anomalies')
display(tp)
print('Top false alarms')
display(fp)
print('Missed anomalies')
display(fn)

display(Image(filename=str(OUT_FULL / 'figures' / 'fig_run_score_boxplot_wc.png')))
display(Image(filename=str(OUT_FULL / 'figures' / 'fig_roc_pr_by_config_wc.png')))


## Research Question: Can Split-Conformal Residual Thresholds Stay Reliable Without Per-Workload Tuning?

This section answers the core reliability question directly from the generated case-level outputs.

We evaluate whether a single benign-calibrated threshold can hold a target false-alarm budget across workloads without tuning thresholds separately for each workload.


In [ ]:
case_pred = pd.read_csv(OUT_FULL / 'case_predictions.csv')
TARGET_ALPHA = 0.05

reliability_rows = []
for cfg, d in case_pred.groupby('config', sort=False):
    benign = d[d['label'] == 0].copy()
    anomaly = d[d['label'] == 1].copy()
    reliability_rows.append({
        'config': cfg,
        'target_alpha': TARGET_ALPHA,
        'benign_block_false_alarm_rate': benign['n_block_alerts'].sum() / benign['n_blocks'].sum(),
        'benign_persist_false_alarm_rate': benign['n_persist_alerts'].sum() / benign['n_blocks'].sum(),
        'benign_run_false_alarm_rate': benign['run_alert'].mean(),
        'anomaly_run_detection_rate': anomaly['run_alert'].mean(),
        'median_anomaly_time_to_detect_s': anomaly.loc[anomaly['run_alert'] == 1, 'time_to_detect_s'].median(),
    })

reliability = pd.DataFrame(reliability_rows)
reliability_by_workload = (
    case_pred[case_pred['label'] == 0]
    .groupby(['config', 'workload'], sort=False)
    .apply(
        lambda x: pd.Series({
            'target_alpha': TARGET_ALPHA,
            'benign_block_false_alarm_rate': x['n_block_alerts'].sum() / x['n_blocks'].sum(),
            'benign_persist_false_alarm_rate': x['n_persist_alerts'].sum() / x['n_blocks'].sum(),
            'benign_run_false_alarm_rate': x['run_alert'].mean(),
        }),
        include_groups=False,
    )
    .reset_index()
)

(OUT_PAPER / 'full').mkdir(parents=True, exist_ok=True)
reliability.to_csv(OUT_PAPER / 'full' / 'conformal_reliability_summary.csv', index=False)
reliability_by_workload.to_csv(OUT_APPENDIX / 'full' / 'conformal_reliability_by_workload.csv', index=False)

print('Conformal reliability summary')
display(reliability)

print('Benign false-alarm rate by workload (no per-workload tuning)')
display(reliability_by_workload)


## Edge Digital-Twin Variants

This table reframes the generated outputs as edge digital-twin variants:
- `Single-head global twin`: one compact benign twin per observation head.
- `Workload-conditioned twin`: same edge model plus workload-conditioned residual scoring.
- `Mechanism-diagnosis twin`: same edge model with grouped mechanism attribution.

This is still edge-friendly because the model family remains linear and compact; the extra complexity is in scoring and diagnosis, not in a large cloud model.


In [ ]:
diag = pd.read_csv(OUT_FULL / 'stressor_diagnosis_metrics.csv')
variant_rows = []
for _, row in overall_full.iterrows():
    cfg = row['config']
    drow = diag[diag['config'] == cfg].iloc[0]
    variant_rows.append({
        'config': cfg,
        'single_head_roc_auc': row['roc_auc'],
        'single_head_pr_auc': row['pr_auc'],
        'workload_conditioned_roc_auc': row['roc_auc_wc'],
        'workload_conditioned_pr_auc': row['pr_auc_wc'],
        'mechanism_top1_acc': drow['top1_acc'],
        'mechanism_top2_acc': drow['top2_acc'],
        'mechanism_macro_f1': drow['macro_f1'],
    })
variant_summary = pd.DataFrame(variant_rows)
variant_summary.to_csv(OUT_PAPER / 'full' / 'digital_twin_variant_summary.csv', index=False)
print('Digital twin variant summary')
display(variant_summary)


## Capability Comparison With Prior Work

This is a capability-based comparison, not an accuracy-based one, because the prior papers use different platforms, observability sources, and datasets.

The table is derived from the attached E-SCOUT, OCTANE, TRACK, and DICE draft materials and is intended for positioning in the paper or appendix.


In [ ]:
prior_work = pd.DataFrame([
    {
        'method': 'E-SCOUT',
        'observability': 'Chip counters / sensors',
        'continuous_test': 'yes',
        'diagnosis_cues': 'partial',
        'update_support': 'not reported',
        'statistical_thresholding': 'outlier scoring',
        'target_false_alarm_control': 'not explicit',
        'runs_at': 'host/edge (+ optional cloud)',
    },
    {
        'method': 'OCTANE',
        'observability': 'Chip counters / sensors (PMU/MSR + sensors)',
        'continuous_test': 'yes',
        'diagnosis_cues': 'partial',
        'update_support': 'partial',
        'statistical_thresholding': 'telemetry anomaly scoring',
        'target_false_alarm_control': 'not explicit',
        'runs_at': 'on-device / on-chip',
    },
    {
        'method': 'TRACK',
        'observability': 'Cross-platform telemetry representations',
        'continuous_test': 'yes',
        'diagnosis_cues': 'partial',
        'update_support': 'partial',
        'statistical_thresholding': 'representation-based scoring',
        'target_false_alarm_control': 'not explicit',
        'runs_at': 'device/host (fleet)',
    },
    {
        'method': 'DICE (this notebook)',
        'observability': 'OS telemetry + OS-mediated proxies + optional profiling',
        'continuous_test': 'yes',
        'diagnosis_cues': 'yes',
        'update_support': 'yes',
        'statistical_thresholding': 'split-conformal residual thresholding',
        'target_false_alarm_control': 'yes, benign-calibrated',
        'runs_at': 'host edge; optional fleet',
    },
])
(OUT_APPENDIX / 'comparison').mkdir(parents=True, exist_ok=True)
prior_work.to_csv(OUT_APPENDIX / 'comparison' / 'prior_work_capability_comparison.csv', index=False)
prior_work.to_csv(OUT_PAPER / 'comparison_prior_work_capability_comparison.csv', index=False)
print('Prior work capability comparison')
display(prior_work)


## Workload/Software Drift Proxy and Appendix Bundles

`results_dice_full_holdout/` is the portable workload-holdout evaluation. In this notebook, it is the practical proxy for workload/software drift robustness.


In [ ]:
if (OUT_HOLDOUT / 'holdout_robustness_summary.csv').exists():
    holdout = pd.read_csv(OUT_HOLDOUT / 'holdout_robustness_summary.csv')
    print('Holdout robustness summary')
    display(holdout)
else:
    print('No holdout summary found yet.')

for folder in [OUT_PAPER, OUT_APPENDIX]:
    print(f'\n{folder}')
    if folder.exists():
        files = sorted(str(p.relative_to(folder)) for p in folder.rglob('*') if p.is_file())
        display(pd.DataFrame({'file': files[:100]}))
    else:
        print('Missing:', folder)


## DICE-Specific Design-Space Evaluation

OCTANE-style ROC/PR sweeps are useful, but DICE should show something more digital-twin specific: how observability, reliability, diagnosis, and detection latency trade off across edge micro-twin heads.

This section builds:
- an observability-performance frontier
- a twin-gain table showing the benefit of workload-conditioned residual scoring
- a stressor-level reliability map for anomaly detectability and time-to-detect


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

paper_full = OUT_PAPER / 'full'
paper_fig = paper_full / 'figures'
paper_full.mkdir(parents=True, exist_ok=True)
paper_fig.mkdir(parents=True, exist_ok=True)

cfg_label = {
    'tier0': 'Tier-0',
    'tier0_tier1': 'Tier-0/1',
    'tier0_tier1_tier2': 'Tier-0/1/2',
}

frontier = (
    overall_full[['config', 'roc_auc', 'pr_auc', 'roc_auc_wc', 'pr_auc_wc']]
    .merge(
        sequential[['config', 'anomaly_detect_rate', 'median_time_to_detect_s', 'detect_within_120s', 'detect_within_300s']],
        on='config',
    )
    .merge(diagnosis[['config', 'top1_acc', 'top2_acc', 'macro_f1']], on='config')
    .merge(
        reliability[['config', 'target_alpha', 'benign_block_false_alarm_rate', 'benign_persist_false_alarm_rate', 'benign_run_false_alarm_rate']],
        on='config',
    )
)
frontier = frontier.merge(
    case_pred.groupby('config', sort=False)['n_features'].median().rename('n_features').reset_index(),
    on='config',
)
if (OUT_HOLDOUT / 'holdout_robustness_summary.csv').exists():
    holdout_frontier = pd.read_csv(OUT_HOLDOUT / 'holdout_robustness_summary.csv')
    holdout_frontier = holdout_frontier.rename(
        columns={
            'mean_pr_auc': 'holdout_mean_pr_auc',
            'worst_pr_auc': 'holdout_worst_pr_auc',
            'mean_roc_auc': 'holdout_mean_roc_auc',
        }
    )
    frontier = frontier.merge(
        holdout_frontier[['config', 'holdout_mean_pr_auc', 'holdout_worst_pr_auc', 'holdout_mean_roc_auc']],
        on='config',
        how='left',
    )
else:
    frontier['holdout_mean_pr_auc'] = np.nan
    frontier['holdout_worst_pr_auc'] = np.nan
    frontier['holdout_mean_roc_auc'] = np.nan
frontier['label'] = frontier['config'].map(cfg_label).fillna(frontier['config'])
frontier['observability_depth'] = frontier['label'].str.count('/') + 1
frontier['wc_gain_roc_auc'] = frontier['roc_auc_wc'] - frontier['roc_auc']
frontier['wc_gain_pr_auc'] = frontier['pr_auc_wc'] - frontier['pr_auc']
frontier['reliability_margin'] = frontier['target_alpha'] - frontier['benign_block_false_alarm_rate']
frontier['joint_detection_diagnosis'] = frontier['anomaly_detect_rate'] * frontier['top1_acc']
frontier['edge_efficiency_pr_per_feature'] = frontier['pr_auc_wc'] / frontier['n_features']
frontier['portable_pr_auc'] = frontier['holdout_mean_pr_auc'].fillna(frontier['pr_auc_wc'])
frontier['portable_roc_auc'] = frontier['holdout_mean_roc_auc'].fillna(frontier['roc_auc_wc'])
frontier.to_csv(paper_full / 'digital_twin_frontier_summary.csv', index=False)

gain = frontier[
    [
        'config',
        'label',
        'wc_gain_roc_auc',
        'wc_gain_pr_auc',
        'reliability_margin',
        'joint_detection_diagnosis',
        'edge_efficiency_pr_per_feature',
    ]
]
gain.to_csv(paper_full / 'digital_twin_gain_summary.csv', index=False)

stress_reliability = (
    case_pred[case_pred['label'] == 1]
    .groupby(['config', 'stressor'], sort=False)
    .apply(
        lambda x: pd.Series(
            {
                'anomaly_detect_rate': x['run_alert'].mean(),
                'median_time_to_detect_s': x.loc[x['run_alert'] == 1, 'time_to_detect_s'].median(),
                'median_peak_block_score': x['peak_block_score'].median(),
            }
        )
        , include_groups=False
    )
    .reset_index()
)
stress_reliability['label'] = stress_reliability['config'].map(cfg_label).fillna(stress_reliability['config'])
stress_reliability.to_csv(paper_full / 'stressor_reliability_map.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
scatter = axes[0].scatter(
    frontier['n_features'],
    frontier['portable_pr_auc'],
    s=frontier['joint_detection_diagnosis'].fillna(0.0) * 1800 + 140,
    c=frontier['reliability_margin'],
    cmap='viridis',
    edgecolor='black',
    linewidth=0.8,
)
for _, row in frontier.iterrows():
    axes[0].annotate(row['label'], (row['n_features'], row['portable_pr_auc']), textcoords='offset points', xytext=(6, 6))
axes[0].set_xlabel('Median active features per edge head')
axes[0].set_ylabel('Portable AUC-PR (holdout if available)')
axes[0].set_title('Observability-portability frontier')
cbar = fig.colorbar(scatter, ax=axes[0])
cbar.set_label('Reliability margin (target alpha - empirical block FAR)')

axes[1].bar(frontier['label'], frontier['wc_gain_pr_auc'], color=['#0f766e', '#1d4ed8', '#b45309'])
axes[1].axhline(0.0, color='black', linewidth=1.0)
axes[1].set_ylabel('AUC-PR gain over single-head twin')
axes[1].set_title('Digital-twin gain from workload-conditioned scoring')

fig.tight_layout()
frontier_png = paper_fig / 'fig_digital_twin_frontier.png'
fig.savefig(frontier_png, dpi=200, bbox_inches='tight')
plt.close(fig)

detect_map = stress_reliability.pivot(index='stressor', columns='label', values='anomaly_detect_rate').fillna(0.0)
ttd_map = stress_reliability.pivot(index='stressor', columns='label', values='median_time_to_detect_s')

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
im0 = axes[0].imshow(detect_map.values, aspect='auto', cmap='YlGn', vmin=0.0, vmax=1.0)
axes[0].set_xticks(range(len(detect_map.columns)), detect_map.columns, rotation=30, ha='right')
axes[0].set_yticks(range(len(detect_map.index)), detect_map.index)
axes[0].set_title('Stressor-level anomaly detect rate')
for i in range(detect_map.shape[0]):
    for j in range(detect_map.shape[1]):
        axes[0].text(j, i, f"{detect_map.iloc[i, j]:.2f}", ha='center', va='center', color='black')
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

ttd_plot = ttd_map.fillna(ttd_map.max().max())
im1 = axes[1].imshow(ttd_plot.values, aspect='auto', cmap='magma_r')
axes[1].set_xticks(range(len(ttd_plot.columns)), ttd_plot.columns, rotation=30, ha='right')
axes[1].set_yticks(range(len(ttd_plot.index)), ttd_plot.index)
axes[1].set_title('Median time-to-detect (s) for detected anomalies')
for i in range(ttd_plot.shape[0]):
    for j in range(ttd_plot.shape[1]):
        value = ttd_map.iloc[i, j]
        text = 'NA' if pd.isna(value) else f"{value:.0f}"
        axes[1].text(j, i, text, ha='center', va='center', color='white')
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

fig.tight_layout()
stressor_png = paper_fig / 'fig_stressor_reliability_map.png'
fig.savefig(stressor_png, dpi=200, bbox_inches='tight')
plt.close(fig)

print('Digital twin frontier summary')
display(frontier[['label', 'n_features', 'portable_pr_auc', 'holdout_worst_pr_auc', 'reliability_margin', 'anomaly_detect_rate', 'median_time_to_detect_s', 'top1_acc', 'joint_detection_diagnosis']])

print('Digital twin gain summary')
display(gain)

print('Stressor-level reliability map')
display(stress_reliability)

display(Image(filename=str(frontier_png)))
display(Image(filename=str(stressor_png)))


## Tier-Correlation Dashboard

This section shows how the observation tiers interact inside the final DICE head. The goal is not only to identify the dominant tier, but also to show whether tiers are redundant, complementary, or stressor-specific in their evidence contribution.


In [ ]:
from itc_notebook_helpers import render_tier_correlation_dashboard

tier_corr, stressor_tier, tier_corr_png = render_tier_correlation_dashboard(OUT_FULL, PAPER_FULL, PAPER_FIG)
print('Tier-share correlation matrix')
display(tier_corr)
print('Mean tier evidence by stressor')
display(stressor_tier)
display(Image(filename=str(tier_corr_png)))


## Residual Evidence Concentration

OCTANE reports how frequently top features are used. For DICE, a more digital-twin-specific question is how much anomaly evidence is explained by the top-k residual contributors.

If most residual evidence is captured by a few features or mechanism groups, the twin is not only accurate but also interpretable and edge-friendly.


In [ ]:
case_diag = pd.read_csv(OUT_FULL / 'case_diagnosis_summary.csv')
anom_diag = case_diag[case_diag['label'] == 1].copy()
mech_cols = [
    'compute_contrib',
    'memory_io_contrib',
    'thermal_power_contrib',
    'scheduler_runtime_contrib',
    'platform_pressure_contrib',
]
anom_diag['total_evidence'] = anom_diag[mech_cols].sum(axis=1).replace(0.0, np.nan)

for k in range(1, 6):
    cols = [f'top_feature_score_{i}' for i in range(1, k + 1)]
    anom_diag[f'feature_top{k}_coverage'] = anom_diag[cols].sum(axis=1) / anom_diag['total_evidence']

for k in range(1, 4):
    cols = [f'top_mechanism_score_{i}' for i in range(1, k + 1)]
    anom_diag[f'mechanism_top{k}_coverage'] = anom_diag[cols].sum(axis=1) / anom_diag['total_evidence']

coverage_cols = [
    'feature_top1_coverage',
    'feature_top2_coverage',
    'feature_top3_coverage',
    'feature_top4_coverage',
    'feature_top5_coverage',
    'mechanism_top1_coverage',
    'mechanism_top2_coverage',
    'mechanism_top3_coverage',
]
evidence_frontier = anom_diag.groupby('config', sort=False)[coverage_cols].mean().reset_index()
evidence_frontier['label'] = evidence_frontier['config'].map(cfg_label).fillna(evidence_frontier['config'])
evidence_frontier.to_csv(paper_full / 'residual_evidence_concentration.csv', index=False)

stressor_evidence = (
    anom_diag[anom_diag['config'] == 'tier0_tier1_tier2']
    .groupby('stressor', sort=False)[
        [
            'feature_top1_coverage',
            'feature_top3_coverage',
            'feature_top5_coverage',
            'mechanism_top1_coverage',
            'mechanism_top2_coverage',
            'mechanism_top3_coverage',
        ]
    ]
    .mean()
    .reset_index()
)
stressor_evidence.to_csv(paper_full / 'stressor_residual_evidence_concentration.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
feature_k = [1, 2, 3, 4, 5]
mechanism_k = [1, 2, 3]
for _, row in evidence_frontier.iterrows():
    axes[0].plot(feature_k, [row[f'feature_top{k}_coverage'] for k in feature_k], marker='o', linewidth=2, label=row['label'])
    axes[1].plot(mechanism_k, [row[f'mechanism_top{k}_coverage'] for k in mechanism_k], marker='o', linewidth=2, label=row['label'])
axes[0].set_xlabel('Top-k residual features')
axes[0].set_ylabel('Mean anomaly evidence coverage')
axes[0].set_xticks(feature_k)
axes[0].set_ylim(0.0, 1.05)
axes[0].set_title('Residual evidence concentration by feature rank')
axes[1].set_xlabel('Top-k mechanism groups')
axes[1].set_ylabel('Mean anomaly evidence coverage')
axes[1].set_xticks(mechanism_k)
axes[1].set_ylim(0.0, 1.05)
axes[1].set_title('Residual evidence concentration by mechanism rank')
axes[1].legend(loc='lower right')
fig.tight_layout()
coverage_png = paper_fig / 'fig_residual_evidence_concentration.png'
fig.savefig(coverage_png, dpi=200, bbox_inches='tight')
plt.close(fig)

print('Residual evidence concentration across digital-twin heads')
display(evidence_frontier)

print('Final-head stressor evidence concentration')
display(stressor_evidence)

display(Image(filename=str(coverage_png)))


## Bootstrap Confidence Intervals

The deployed DICE head remains compact. The extra runtime is spent offline here, through bootstrap confidence intervals that make the reported metrics more defensible for the paper and appendix.


In [ ]:
from itc_notebook_helpers import (
    export_llm_case_cards,
    render_bootstrap_confidence,
    render_explainability_dashboard,
    render_paper_performance_stack,
    render_portability_dashboard,
)

BOOTSTRAP_SAMPLES = 1000
bootstrap_ci, bootstrap_png = render_bootstrap_confidence(case_pred, PAPER_FULL, PAPER_FIG, samples=BOOTSTRAP_SAMPLES, seed=0)
print('Bootstrap confidence intervals')
display(bootstrap_ci)
display(Image(filename=str(bootstrap_png)))


## Paper Figure Layout

This section assembles the previous results into three reviewer-facing composite figures:
- performance stack: separation, ranking, calibration, and time-to-detect
- explainability dashboard: tier attribution, mechanism attribution, and diagnosis confusion
- portability dashboard: observability frontier, holdout robustness, and bootstrap uncertainty


In [ ]:
performance_stack, performance_stack_png = render_paper_performance_stack(
    case_pred,
    overall_full,
    sequential,
    reliability,
    PAPER_FULL,
    PAPER_FIG,
)
tier_summary, mechanism_summary, cm_summary, explainability_png = render_explainability_dashboard(
    OUT_FULL,
    PAPER_FULL,
    PAPER_FIG,
)
if 'holdout' not in globals() or holdout.empty:
    holdout = pd.read_csv(OUT_HOLDOUT / 'holdout_robustness_summary.csv')
portability_summary, portability_png = render_portability_dashboard(
    frontier,
    holdout,
    bootstrap_ci,
    PAPER_FULL,
    PAPER_FIG,
)

print('Performance stack summary')
display(performance_stack)
print('Portability summary')
display(portability_summary)
display(Image(filename=str(performance_stack_png)))
display(Image(filename=str(explainability_png)))
display(Image(filename=str(portability_png)))


## Optional LLM-Ready Case Cards

The detector itself is not an LLM. DICE remains a benign-trained digital twin with residual scoring and conformal decisioning.

This export is only a structured appendix artifact for optional later-stage diagnostics or reviewer summaries. It is model-agnostic. If you want a local summarizer later, suitable small instruct models include `Qwen2.5-7B-Instruct` or `Llama-3.1-8B-Instruct`, grounded only on the exported case cards.


In [ ]:
llm_cards = export_llm_case_cards(OUT_FULL, APPENDIX_FULL)
print('LLM-ready reviewer cards')
display(llm_cards.head(10))


In [ ]:
manifest = json.loads(MANIFEST.read_text())
print('Manifest path:', MANIFEST)
print('Dataset SHA256:', manifest['dataset_digest']['sha256'])
print('Environment file SHA256:', manifest['environment_files']['environment_yml']['sha256'])
print('Requirements SHA256:', manifest['environment_files']['requirements_txt']['sha256'])
if NOTEBOOK_RUNTIME.exists():
    runtime_summary = json.loads(NOTEBOOK_RUNTIME.read_text())
    print('Notebook runtime summary:', runtime_summary)
print('Main paper artifacts:', PAPER_FULL)
print('Appendix artifacts :', APPENDIX_FULL)
